# HvGlu3 - CESA center-of-mass distance over time

Converts the visual VMD impression ("close in T1, far in T2, moderate in
T3/T4") into an explicit number per trajectory: the center-of-mass (COM)
distance between HvGlu3 (chain A) and the CESA trimer (chains B+C+D),
computed every frame, using ALL atoms (not just backbone) for a more
accurate center of mass.

Run for **both** CESA3 (T1-T4) and CESA1 (8 replicas) so the two systems
can be directly compared.

**What to look for:**
- Does each trajectory's distance plateau (flat) in the last 100-200 ns,
  or is it still trending at the final frame?
- A flat plateau supports "this is a real, stable pose."
- Still-trending at the end means that run hasn't converged yet, and its
  current distance shouldn't be treated as a final answer.

**Outputs (written next to this notebook, one set per system):**
- `hvglu3_cesa_distance_<SYSTEM>.csv` — per-frame distance, per trajectory
- `hvglu3_cesa_distance_<SYSTEM>_summary.csv` — mean/std/min/max, plus
  last-20%-of-trajectory mean (a simple plateau check)
- `hvglu3_cesa_distance_<SYSTEM>.png` — distance vs time, one line per
  trajectory

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda

print('MDAnalysis', mda.__version__)

## Reusable function — works for either system

Takes a topology (PDB, with chain IDs) and a list of trajectory files,
and computes COM(chain A) <-> COM(chains B+C+D) distance per frame.

In [ ]:
def compute_hvglu3_cesa_distance(top_pdb, traj_paths, hvglu3_chain='A',
                                   cesa_chains=('B', 'C', 'D')):
    """Returns dict: trajectory label -> (time_ns array, distance_A array)."""
    results = {}
    cesa_sel = ' or '.join(f'segid {c}' for c in cesa_chains)

    for label, traj_path in traj_paths.items():
        u = mda.Universe(str(top_pdb), str(traj_path))

        hvglu3 = u.select_atoms(f'segid {hvglu3_chain}')
        cesa   = u.select_atoms(cesa_sel)

        if len(hvglu3) == 0 or len(cesa) == 0:
            # Some PDBs use chainID instead of segid - retry with chainID
            hvglu3 = u.select_atoms(f'chainID {hvglu3_chain}')
            cesa_sel_alt = ' or '.join(f'chainID {c}' for c in cesa_chains)
            cesa = u.select_atoms(cesa_sel_alt)

        assert len(hvglu3) > 0, f'{label}: no atoms found for HvGlu3 (chain {hvglu3_chain})'
        assert len(cesa) > 0, f'{label}: no atoms found for CESA chains {cesa_chains}'

        times, dists = [], []
        for ts in u.trajectory:
            com_hvglu3 = hvglu3.center_of_mass()
            com_cesa   = cesa.center_of_mass()
            d = np.linalg.norm(com_hvglu3 - com_cesa)
            times.append(ts.time / 1000.0)   # ps -> ns
            dists.append(d)

        t_ns = np.array(times)
        t_ns = t_ns - t_ns[0]   # reset each trajectory's clock to 0
        results[label] = (t_ns, np.array(dists))

        print(f'  {label}: {len(dists)} frames, '
              f'{t_ns[-1]:.1f} ns, '
              f'mean dist={np.mean(dists):.2f} A, '
              f'last-20% mean={np.mean(dists[int(0.8*len(dists)):]):.2f} A')

    return results


def summarize_and_plot(results, out_csv_frames, out_csv_summary, out_png,
                        title, colors=None):
    """Shared summary table + plot logic for either system."""
    long_rows = []
    for label, (t, d) in results.items():
        for ti, di in zip(t, d):
            long_rows.append({'trajectory': label, 'time_ns': ti, 'distance_A': di})
    df_frames = pd.DataFrame(long_rows)
    df_frames.to_csv(out_csv_frames, index=False)
    print(f'Wrote {out_csv_frames.name}')

    summary_rows = []
    for label, (t, d) in results.items():
        last20 = d[int(0.8 * len(d)):]
        first20 = d[:int(0.2 * len(d))]
        summary_rows.append({
            'trajectory'      : label,
            'n_frames'        : len(d),
            'mean_A'          : float(np.mean(d)),
            'std_A'           : float(np.std(d)),
            'min_A'           : float(np.min(d)),
            'max_A'           : float(np.max(d)),
            'first20pct_mean_A': float(np.mean(first20)),
            'last20pct_mean_A' : float(np.mean(last20)),
            'last20pct_std_A'  : float(np.std(last20)),
            'drift_first_to_last_A': float(np.mean(last20) - np.mean(first20)),
        })
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(out_csv_summary, index=False)
    print(f'Wrote {out_csv_summary.name}')

    fig, ax = plt.subplots(figsize=(10, 4.5))
    cmap = plt.cm.tab10
    for i, (label, (t, d)) in enumerate(results.items()):
        c = colors[label] if colors else cmap(i % 10)
        ax.plot(t, d, label=label, color=c, lw=1.0, alpha=0.85)

    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('HvGlu3-CESA COM distance (A)')
    ax.set_title(title)
    ax.legend(loc='best', frameon=False, ncol=4, fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    print(f'Wrote {out_png.name}')
    plt.show()

    return df_frames, df_summary

## 1. CESA3 — T1, T2, T3, T4

In [ ]:
CESA3_BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')
CESA3_REF_PDB = CESA3_BASE / 'R1' / 'backbone.pdb'

CESA3_TRAJ_PATHS = {
    'T1': CESA3_BASE / 'R1' / 'T1' / 'backbone.xtc',
    'T2': CESA3_BASE / 'R1' / 'T2' / 'backbone.xtc',
    'T3': CESA3_BASE / 'R1' / 'T3' / 'backbone.xtc',
    'T4': CESA3_BASE / 'R1' / 'T4' / 'backbone.xtc',
}

assert CESA3_REF_PDB.exists(), f'Not found: {CESA3_REF_PDB}'
for label, p in CESA3_TRAJ_PATHS.items():
    assert p.exists(), f'Missing {label}: {p}'
    print(f'  {label}: {p.relative_to(CESA3_BASE)}')

In [ ]:
print('Computing CESA3 HvGlu3-CESA3 distance per trajectory...')
cesa3_results = compute_hvglu3_cesa_distance(
    CESA3_REF_PDB, CESA3_TRAJ_PATHS,
    hvglu3_chain='A', cesa_chains=('B', 'C', 'D'),
)

In [ ]:
cesa3_colors = {'T1': '#1f77b4', 'T2': '#2ca02c', 'T3': '#d62728', 'T4': '#9467bd'}

df_cesa3_frames, df_cesa3_summary = summarize_and_plot(
    cesa3_results,
    out_csv_frames =CESA3_BASE / 'hvglu3_cesa_distance_CESA3.csv',
    out_csv_summary=CESA3_BASE / 'hvglu3_cesa_distance_CESA3_summary.csv',
    out_png        =CESA3_BASE / 'hvglu3_cesa_distance_CESA3.png',
    title='CESA3 — HvGlu3-to-CESA3 trimer COM distance (T1-T4)',
    colors=cesa3_colors,
)
df_cesa3_summary.round(2)

## 2. CESA1 — 8 replicas (traj1 ... traj8)

EDIT THE PATHS BELOW to match your actual CESA1 folder/filenames before
running this section.

In [ ]:
CESA1_BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA1/result_CESA1_ClusPro/CESA1_prod/Backbone_xtc')           # <-- EDIT THIS
CESA1_REF_PDB = CESA1_BASE / 'backbone.pdb'                 # <-- EDIT IF DIFFERENT

CESA1_TRAJ_PATHS = {
    f'traj{i}': CESA1_BASE / f'traj{i}.xtc' for i in range(1, 9)
}   # <-- EDIT filenames above if yours differ (e.g. traj1_backbone.xtc)

assert CESA1_REF_PDB.exists(), f'Not found: {CESA1_REF_PDB}'
for label, p in CESA1_TRAJ_PATHS.items():
    assert p.exists(), f'Missing {label}: {p}'
    print(f'  {label}: {p.name}')

In [ ]:
print('Computing CESA1 HvGlu3-CESA1 distance per trajectory...')
cesa1_results = compute_hvglu3_cesa_distance(
    CESA1_REF_PDB, CESA1_TRAJ_PATHS,
    hvglu3_chain='A', cesa_chains=('B', 'C', 'D'),
)

In [ ]:
df_cesa1_frames, df_cesa1_summary = summarize_and_plot(
    cesa1_results,
    out_csv_frames =CESA1_BASE / 'hvglu3_cesa_distance_CESA1.csv',
    out_csv_summary=CESA1_BASE / 'hvglu3_cesa_distance_CESA1_summary.csv',
    out_png        =CESA1_BASE / 'hvglu3_cesa_distance_CESA1.png',
    title='CESA1 — HvGlu3-to-CESA1 trimer COM distance (8 replicas)',
)
df_cesa1_summary.round(2)

## 3. Side-by-side comparison

Puts the two systems' summary stats next to each other. Look especially
at the spread of `mean_A` across trajectories within each system — a
tight spread (CESA1-like) vs. a wide spread (CESA3-like) is the
quantitative version of "consistent in CESA1, inconsistent in CESA3."

In [ ]:
print('CESA3 — per-trajectory mean distance (A):')
print(df_cesa3_summary[['trajectory', 'mean_A', 'last20pct_mean_A', 'drift_first_to_last_A']]
      .round(2).to_string(index=False))

print(f'\nCESA3 spread across trajectories (max-min of mean_A): '
      f'{df_cesa3_summary["mean_A"].max() - df_cesa3_summary["mean_A"].min():.2f} A')

print('\nCESA1 — per-trajectory mean distance (A):')
print(df_cesa1_summary[['trajectory', 'mean_A', 'last20pct_mean_A', 'drift_first_to_last_A']]
      .round(2).to_string(index=False))

print(f'\nCESA1 spread across trajectories (max-min of mean_A): '
      f'{df_cesa1_summary["mean_A"].max() - df_cesa1_summary["mean_A"].min():.2f} A')

## 4. How to read the `drift_first_to_last_A` column

This is `last20pct_mean_A - first20pct_mean_A` for each trajectory —
how much the distance changed between the start and end of the run.

- Close to 0 -> the distance was already stable early on and stayed there
- Far from 0 (either direction) -> that trajectory was still actively
  approaching or moving away from its current distance when the
  simulation ended, i.e. it likely has NOT converged

A trajectory with large drift should be treated cautiously — its final
"pose" may not be a true stable state, just where it happened to be when
you stopped recording.

In [ ]:
CESA1_BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA1/result_CESA1_ClusPro/CESA1_prod/Backbone_xtc')        # <-- EDIT THIS
CESA3_BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')

CESA1_CSV = CESA1_BASE / 'hvglu3_cesa_distance_CESA1.csv'
CESA3_CSV = CESA3_BASE / 'hvglu3_cesa_distance_CESA3.csv'

EXCLUDED_CESA1 = ['traj4', 'traj5']   # outliers — large first-to-last drift

assert CESA1_CSV.exists(), f'Not found: {CESA1_CSV} — run hvglu3_cesa_distance.ipynb first'
assert CESA3_CSV.exists(), f'Not found: {CESA3_CSV} — run hvglu3_cesa_distance.ipynb first'

df_cesa1 = pd.read_csv(CESA1_CSV)
df_cesa3 = pd.read_csv(CESA3_CSV)

print(f'CESA1 trajectories found: {sorted(df_cesa1["trajectory"].unique())}')
print(f'CESA3 trajectories found: {sorted(df_cesa3["trajectory"].unique())}')

In [ ]:
def summary_from_frames(df_frames):
    rows = []
    for traj, g in df_frames.groupby('trajectory'):
        g = g.sort_values('time_ns')
        d = g['distance_A'].values
        n = len(d)
        first20 = d[:int(0.2 * n)]
        last20  = d[int(0.8 * n):]
        rows.append({
            'trajectory'           : traj,
            'n_frames'             : n,
            'mean_A'               : float(np.mean(d)),
            'std_A'                : float(np.std(d)),
            'first20pct_mean_A'    : float(np.mean(first20)),
            'last20pct_mean_A'     : float(np.mean(last20)),
            'drift_first_to_last_A': float(np.mean(last20) - np.mean(first20)),
        })
    return pd.DataFrame(rows)

summary_cesa1_all = summary_from_frames(df_cesa1)
summary_cesa3_all = summary_from_frames(df_cesa3)

In [ ]:
summary_cesa1_filtered = summary_cesa1_all[
    ~summary_cesa1_all['trajectory'].isin(EXCLUDED_CESA1)
].reset_index(drop=True)

print('CESA1 — ALL 8 replicas:')
print(summary_cesa1_all[['trajectory', 'mean_A', 'last20pct_mean_A', 'drift_first_to_last_A']]
      .round(2).to_string(index=False))
spread_all = summary_cesa1_all['mean_A'].max() - summary_cesa1_all['mean_A'].min()
print(f'Spread (max-min of mean_A), all 8: {spread_all:.2f} A\n')

print(f'CESA1 — FILTERED (excluding {EXCLUDED_CESA1}):')
print(summary_cesa1_filtered[['trajectory', 'mean_A', 'last20pct_mean_A', 'drift_first_to_last_A']]
      .round(2).to_string(index=False))
spread_filtered = summary_cesa1_filtered['mean_A'].max() - summary_cesa1_filtered['mean_A'].min()
print(f'Spread (max-min of mean_A), filtered 6: {spread_filtered:.2f} A')

In [ ]:
spread_cesa3 = summary_cesa3_all['mean_A'].max() - summary_cesa3_all['mean_A'].min()

print('CESA3 — T1-T4:')
print(summary_cesa3_all[['trajectory', 'mean_A', 'last20pct_mean_A', 'drift_first_to_last_A']]
      .round(2).to_string(index=False))
print(f'CESA3 spread (max-min of mean_A): {spread_cesa3:.2f} A\n')

print('=' * 60)
print(f'CESA3 spread (4 trajectories)            : {spread_cesa3:.2f} A')
print(f'CESA1 spread (all 8 replicas)             : {spread_all:.2f} A')
print(f'CESA1 spread (6 replicas, traj4/5 removed): {spread_filtered:.2f} A')
print('=' * 60)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import MDAnalysis as mda

BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')
REF_PDB = BASE / 'R1' / 'backbone.pdb'
TRAJ_PATHS = {
    'T1': BASE / 'R1' / 'T1' / 'backbone.xtc',
    'T2': BASE / 'R1' / 'T2' / 'backbone.xtc',
    'T3': BASE / 'R1' / 'T3' / 'backbone.xtc',
    'T4': BASE / 'R1' / 'T4' / 'backbone.xtc',
}

rows = []
for label, path in TRAJ_PATHS.items():
    u = mda.Universe(str(REF_PDB), str(path))
    hvglu3 = u.select_atoms('segid A')
    chainB = u.select_atoms('segid B')
    chainC = u.select_atoms('segid C')
    chainD = u.select_atoms('segid D')

    for ts in u.trajectory:
        com_a = hvglu3.center_of_mass()
        rows.append({
            'trajectory': label,
            'frame_in_traj': ts.frame,
            'time_ns': ts.time / 1000.0,
            'dist_B': np.linalg.norm(com_a - chainB.center_of_mass()),
            'dist_C': np.linalg.norm(com_a - chainC.center_of_mass()),
            'dist_D': np.linalg.norm(com_a - chainD.center_of_mass()),
        })
    print(f'{label} done: {len(u.trajectory)} frames')

df = pd.DataFrame(rows)
df.to_csv(BASE / 'hvglu3_persubunit_distance_CESA3.csv', index=False)
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')
df_dist = pd.read_csv(BASE / 'hvglu3_persubunit_distance_CESA3.csv')
df_clusters = pd.read_csv(BASE / 'KMeans' / 'PCA_clusters.csv')

print(f'df_dist total rows: {len(df_dist)}')
print(f'df_clusters total rows: {len(df_clusters)}')

# Real per-trajectory lengths (confirmed via gmx check)
traj_lengths = {'T1': 401, 'T2': 417, 'T3': 420, 'T4': 409}

# Each junction drops 1 duplicate frame (first frame of T2/T3/T4 matches
# last frame of previous trajectory) - confirmed pattern from earlier RMSD work
offsets = {}
cumulative = 0
for i, (traj, length) in enumerate(traj_lengths.items()):
    if i == 0:
        offsets[traj] = 0
        cumulative = length
    else:
        offsets[traj] = cumulative  # this trajectory's frame 1 (0-indexed) starts here, since frame 0 was the dropped duplicate
        cumulative += (length - 1)  # subtract the 1 dropped duplicate frame

print('Offsets into merged file:', offsets)
print(f'Total merged frames implied: {cumulative}')

In [ ]:
# For each trajectory, drop its first frame if it's not T1 (since that frame
# is the duplicate that gets removed during merging), then compute the
# correct merged_frame_idx for the remaining frames

merged_rows = []
for traj, length in traj_lengths.items():
    sub = df_dist[df_dist['trajectory'] == traj].sort_values('frame_in_traj').reset_index(drop=True)
    if traj != 'T1':
        sub = sub.iloc[1:].reset_index(drop=True)  # drop the duplicate first frame
    sub['merged_frame_idx'] = offsets[traj] + np.arange(len(sub))
    merged_rows.append(sub)

df_dist_merged = pd.concat(merged_rows, ignore_index=True)
print(f'Total after dedup: {len(df_dist_merged)}')  # should be 1644

# Join with cluster assignments
df_dist_merged = df_dist_merged.merge(
    df_clusters[['Cluster']], left_on='merged_frame_idx', right_index=True, how='left'
)

print(df_dist_merged[['trajectory', 'merged_frame_idx', 'Cluster', 'dist_B', 'dist_C', 'dist_D']].head())
print(f'\nAny unmatched (NaN cluster)? {df_dist_merged["Cluster"].isna().sum()}')

In [ ]:
com_by_cluster = df_dist_merged.groupby('Cluster')[['dist_B', 'dist_C', 'dist_D']].mean()
com_by_cluster.index = com_by_cluster.index + 1  # 1-indexed for readability
print(com_by_cluster.round(2))

In [ ]:
for cluster_num in range(1, 5):
    # using whatever variable holds each cluster's contact dataframe
    print(f"Cluster {cluster_num} chain breakdown (>50%):")
    print(cluster_contacts_df[cluster_contacts_df['frequency'] >= 0.5]['CESA3_chain'].value_counts())